# BTProp – Đánh giá mức độ ảo giác của AI khi sinh câu hỏi trắc nghiệm

Cài đặt lại phương pháp **Belief Tree Propagation** (Hou et al., NAACL 2025 – [paper](https://aclanthology.org/2025.naacl-long.158.pdf), [code gốc](https://github.com/UCSB-NLP-Chang/BTProp)) và **nâng cấp** để nhận đầu vào là một câu hỏi trắc nghiệm dạng JSON:

```json
{
  "content": "string",
  "options": [{"key": "string", "text": "string", "is_correct": "string"}],
  "explanation": "string"
}
```

Chương trình in ra **mức độ ảo giác** (hallucination score ∈ [0, 1]) của AI khi tạo ra câu hỏi đó.

**Pipeline**
1. Chuyển câu hỏi + từng đáp án thành mệnh đề khẳng định; tách `explanation` thành các claim nguyên tử.
2. Với mỗi mệnh đề: xây *belief tree* (decomposition / supportive–contradictory premises / statement correction), lấy confidence của LLM, gán quan hệ logic bằng NLI.
3. Suy luận posterior $p(Z_0 = T \mid \{S_u\})$ bằng *hidden Markov tree* (belief propagation).
4. Tổng hợp thành điểm ảo giác của cả câu hỏi.

**Yêu cầu**: Runtime GPU (T4 trở lên). Backbone mặc định `Qwen/Qwen2.5-7B-Instruct` nạp 4-bit.


In [ ]:
#@title 1. Cài đặt thư viện
!pip -q install -U "transformers>=4.44" accelerate bitsandbytes sentencepiece protobuf
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


In [ ]:
#@title 2. Cấu hình & nạp mô hình
import json, re, math, itertools
from typing import List, Optional, Dict, Tuple
import torch
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          AutoModelForSequenceClassification, BitsAndBytesConfig)

LLM_NAME   = "Qwen/Qwen2.5-7B-Instruct"   #@param ["Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-3B-Instruct", "meta-llama/Meta-Llama-3-8B-Instruct"]
LOAD_4BIT  = True                         #@param {type:"boolean"}
NLI_NAME   = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"  #@param ["MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7", "cross-encoder/nli-deberta-v3-large"]
MAX_LAYERS = 2                            #@param {type:"integer"}
VERBOSE    = True                         #@param {type:"boolean"}
FORCE_BOTH_STRATEGIES = True              #@param {type:"boolean"}
FILTER_LANGUAGE       = True              #@param {type:"boolean"}
PIVOT_TO_ENGLISH      = True              #@param {type:"boolean"}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
quant = None
if LOAD_4BIT and DEVICE == "cuda":
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=torch.float16)

tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME, quantization_config=quant, torch_dtype=torch.float16, device_map="auto")
llm_model.eval()
print("LLM loaded:", LLM_NAME)


In [ ]:
#@title 3. Prompts (chuyển thể từ Appendix A.7 của paper)
LANG_RULE = ("Write every output sentence in the SAME language as the input statement "
             "(if the statement is Vietnamese, answer in Vietnamese; never switch to German, French, Chinese, etc.).")

CONFIDENCE_PROMPT = """Are the following statements true or false? For each of the following statements, determine whether it is true or false. Provide a response of 'True' if the statement is correct, or 'False' if the statement is incorrect.
Remember:
- Judge using your own knowledge only; you do not have access to external data.
- Well-known facts, including those with specific numbers, dates or names, should be judged True if they match your knowledge.
- If the statement is vague, unverifiable, or contradicts your knowledge, answer False.
- Directly output "True" or "False" without adding any markers."""

DECOMPOSE_PROMPT = """**Fact-Checking Claims Extraction:**
**Objective:** Analyze the provided statement to extract and list all distinct factual claims that require verification. Each listed claim should be verifiable and not overlap in content with other claims listed.
**Instructions:**
1. Identify parts of the statement that assert specific, verifiable facts. In every claim, always use the **full names** when referring to any concept, person, or entity. **Avoid the use of pronouns or indirect references** that require contextual understanding.
2. List each verifiable claim separately. Ensure that each claim is distinct and there is no overlap in the factual content between any two claims.
3. Exclude any statements that are purely hypothetical, speculative, or subjective opinions.
4. If the statement does not contain any verifiable facts, output: "Claim 1: No check-worthy claims available."
5. """ + LANG_RULE + """
**Output Format**:
Claim 1: <the first claim>
Claim 2: <the second claim>
...
**Examples:**
Statement: According to recent data, China has surpassed the United States in terms of GDP when measured using Purchasing Power Parity (PPP), and India is projected to overtake China by 2030.
Claim 1: China has surpassed the United States in terms of GDP when measured using Purchasing Power Parity (PPP).
Claim 2: India is projected to overtake China in terms of GDP by 2030.

Statement: The world's largest desert is Antarctica, and it is larger than the Sahara.
Claim 1: The world's largest desert is Antarctica.
Claim 2: Antarctica is larger than the Sahara.

Statement: I think pizza is the best food ever!
Claim 1: No check-worthy claims available.

Statement: The software 'Photoshop' was released by Adobe Systems in 1988.
Claim 1: The software 'Photoshop' was released by Adobe Systems in 1988."""

SUPPORT_PROMPT = """**Finding Supportive Premises**
Is the following statement true or false? If it is true, list several supportive premises for it.
**Important Rules:**
1. Each premise should be clearly stated and directly relevant to the target statement.
2. Do not use pronouns in generated premises. Always use the full name of each person, event, object, etc.
3. """ + LANG_RULE + """
**Output Format**:
Judgement: <True or False>
Premise 1: <the first premise>
Premise 2: <the second premise>
...
If the statement is false, simply output:
Judgement: False
Premise 1: No supportive premises applicable.
**Examples:**
Target statement: Renewable energy sources will lead to a decrease in global greenhouse gas emissions.
Judgement: True
Premise 1: Renewable energy sources produce electricity without emitting carbon dioxide.
Premise 2: Increasing the adoption of renewable energy reduces reliance on fossil fuels, which are the primary source of industrial carbon dioxide emissions.

Target statement: Eating carrots improves night vision.
Judgement: False
Premise 1: No supportive premises applicable."""

CONTRADICT_PROMPT = """**Finding Contradictory Premises**
Is the following statement true or false? If it is false, list several contradictory premises for it.
**Important Rules:**
1. Each premise should be clearly stated and directly relevant to the target statement.
2. Do not use pronouns in generated premises. Always use the full name of each person, event, object, etc.
3. """ + LANG_RULE + """
**Output Format**:
Judgement: <True or False>
Premise 1: <the first premise>
Premise 2: <the second premise>
...
If the statement is true, simply output:
Judgement: True
Premise 1: No contradictory premises applicable.
**Examples:**
Target statement: Renewable energy sources will lead to a decrease in global greenhouse gas emissions.
Judgement: True
Premise 1: No contradictory premises applicable.

Target statement: Eating carrots improves night vision.
Judgement: False
Premise 1: The belief that eating carrots improves night vision stems from World War II propaganda, not from scientific evidence.
Premise 2: While carrots are rich in vitamin A, which is necessary for maintaining healthy vision, they do not enhance night vision beyond normal levels."""

QUESTION_PROMPT = """Given a statement, your task is to identify a general question that can be used to check the truthfulness of the statement. The question should directly address the claim to confirm or refute it without seeking additional detailed information. """ + LANG_RULE + """ Output only one line starting with "Question: ".
**Examples:**
Statement: Thủ đô của Pháp là thành phố Berlin.
Question: Thủ đô của Pháp là thành phố nào?

Statement: Bitcoin was created in 2009 by an anonymous entity known as Satoshi Nakamoto.
Question: Who created Bitcoin in 2009?

Statement: The iPhone was first released by Apple in 2007.
Question: When was iPhone first released by Apple?

Statement: The Eiffel Tower is located in Paris.
Question: Where is the Eiffel Tower located?

Statement: Albert Einstein was a physicist.
Question: What profession was Albert Einstein?"""

REVISE_PROMPT = """**Background Knowledge**: {background}
Leverage the above provided knowledge and your own knowledge to review the correctness of the following statement:
**Statement**: {statement}
Instruction:
- If the statement is correct, output it unchanged.
- If the statement is **not mentioned in the background knowledge and its correctness cannot be determined**, you should also directly output the statement **unchanged**.
- If the statement is wrong, revise only the parts of the statement that are incorrect, to align with the background knowledge. Do not add any additional sentences or details.
- """ + LANG_RULE + """
**Output Format:**
Revised Answer: <the original statement if it is correct or not mentioned; the revised statement if it is inaccurate>
**Examples:**
Background Knowledge: Thủ đô của Pháp là Paris.
Statement: Thủ đô của Pháp là thành phố Berlin.
Revised Answer: Thủ đô của Pháp là thành phố Paris.

Background Knowledge: Bitcoin was created by Satoshi Nakamoto in 2009.
Statement: Bitcoin was created in 2009 by Elon Musk.
Revised Answer: Bitcoin was created in 2009 by Satoshi Nakamoto.

Background Knowledge: The Seine river flows through the city of Paris.
Statement: Paris is the capital of France.
Revised Answer: Paris is the capital of France."""

STRATEGY_PROMPT = """**Task: Choose the Best Strategy for Premise Generation**
We need to generate several premises for a given target statement. We have 2 techniques:
1. Logical Relationships: creating premises based on entailment or contradiction.
2. Statement Perturbation: create variations of the statement by altering key details (names, numbers, dates, places) to form contradictory premises.
**Important Guidelines**:
1. Prioritize logical relationships when it is straightforward to find supportive or contradictory premises.
2. If a statement contains particular names, numbers, timestamps, or other conditions that can be varied, consider statement perturbation.
3. If both methods are applicable, output "both".
**Output Format:** one of "Logical Relation", "Statement Perturbation", "both".
Target statement: Eating a balanced diet improves overall health.
Output: Logical Relation
Target statement: 'War and Peace' was a book written by Leo Tolstoy.
Output: Statement Perturbation
Target statement: Water boils at 100 degrees Celsius at sea level
Output: both
Target statement: Mount Everest is 8,848 meters tall.
Output: Statement Perturbation
Target statement: Artificial intelligence will replace most human jobs in the future
Output: Logical Relation"""

TRANSLATE_PROMPT = """Translate the following sentence into English. Keep all names, numbers and facts exactly as they are; do not add, remove or correct anything. Output only one line starting with "English: ".
Sentence: Thủ đô của Pháp là thành phố Berlin.
English: The capital of France is the city of Berlin.

Sentence: Sao Mộc là hành tinh lớn nhất Hệ Mặt Trời, có khối lượng gấp khoảng 318 lần Trái Đất.
English: Jupiter is the largest planet in the Solar System, with a mass about 318 times that of Earth."""

MCQ_TO_STATEMENT_PROMPT = """Rewrite a multiple-choice question together with ONE answer option into a single, self-contained declarative statement that asserts this option is the answer to the question.
**Important rules:**
- Keep the option's content EXACTLY as given, even if it is factually wrong. NEVER replace the option with the correct answer or fix its numbers/names/dates.
- Do not judge whether the statement is true. Do not use pronouns. """ + LANG_RULE + """
- Output only one line starting with "Statement: ".
**Examples:**
Question: What is the capital of France?
Option: Berlin
Statement: The capital of France is Berlin.

Question: At what temperature does water boil at standard atmospheric pressure?
Option: 90°C
Statement: Water boils at 90°C at standard atmospheric pressure.

Question: Which planet is known as the Red Planet?
Option: Mars
Statement: Mars is the planet known as the Red Planet.

Question: Thủ đô của Việt Nam là thành phố nào?
Option: Hà Nội
Statement: Thủ đô của Việt Nam là thành phố Hà Nội."""


In [ ]:
#@title 4. Bọc LLM: sinh văn bản & ước lượng confidence (prior belief)
class LLM:
    def __init__(self, model, tokenizer):
        self.model, self.tok = model, tokenizer
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.true_ids  = self._first_ids(["True", " True", "true", " true", "TRUE"])
        self.false_ids = self._first_ids(["False", " False", "false", " false", "FALSE"])
        self.cache: Dict[str, float] = {}

    def _first_ids(self, words):
        ids = set()
        for w in words:
            t = self.tok.encode(w, add_special_tokens=False)
            if t:
                ids.add(t[0])
        return sorted(ids)

    def _chat(self, messages):
        return self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    @torch.no_grad()
    def generate(self, prompt: str, temperature=0.0, max_tokens=256, n=1) -> List[str]:
        text = self._chat([{"role": "user", "content": prompt}])
        enc = self.tok(text, return_tensors="pt").to(self.model.device)
        kw = dict(max_new_tokens=max_tokens, pad_token_id=self.tok.pad_token_id)
        if temperature > 0:
            kw.update(do_sample=True, temperature=temperature, top_p=0.95, num_return_sequences=n)
        else:
            kw.update(do_sample=False, num_return_sequences=1)
        out = self.model.generate(**enc, **kw)
        gen = out[:, enc.input_ids.shape[1]:]
        return [self.tok.decode(g, skip_special_tokens=True).strip() for g in gen]

    @torch.no_grad()
    def confidence(self, statement: str) -> float:
        """p(True) từ phân phối token kế tiếp, chuẩn hoá giữa 'True' và 'False' (Sec. 3.2 - Prior Belief Estimation)."""
        statement = statement.strip()
        if statement in self.cache:
            return self.cache[statement]
        text = self._chat([{"role": "user", "content": CONFIDENCE_PROMPT + "\n\nStatement: " + statement}])
        enc = self.tok(text, return_tensors="pt").to(self.model.device)
        logits = self.model(**enc).logits[0, -1].float()
        probs = torch.softmax(logits, -1)
        p_t = probs[self.true_ids].sum().item()
        p_f = probs[self.false_ids].sum().item()
        if (p_t + p_f) < 0.2:
            # token đầu không phải True/False (vd. model mở đầu bằng từ khác) -> sinh ngắn rồi parse
            ans = self.generate(CONFIDENCE_PROMPT + "\n\nStatement: " + statement, temperature=0.0, max_tokens=6)[0].lower()
            conf = 0.9 if "true" in ans and "false" not in ans else 0.1 if "false" in ans else 0.5
        else:
            conf = p_t / (p_t + p_f)
        self.cache[statement] = conf
        return conf

llm = LLM(llm_model, tok)
print("Sanity check  ->  conf('Paris is the capital of France') =", round(llm.confidence("Paris is the capital of France."), 3),
      "| conf('Berlin is the capital of France') =", round(llm.confidence("Berlin is the capital of France."), 3))


In [ ]:
#@title 5. Mô hình NLI gán quan hệ logic giữa node cha – con
class NLI:
    def __init__(self, name: str, device: str):
        self.tok = AutoTokenizer.from_pretrained(name)
        self.model = AutoModelForSequenceClassification.from_pretrained(name).to(device).eval()
        self.device = device
        self.id2label = {i: l.lower() for i, l in self.model.config.id2label.items()}

    @torch.no_grad()
    def _pred(self, pairs: List[Tuple[str, str]]) -> List[str]:
        enc = self.tok([p for p, _ in pairs], [h for _, h in pairs], return_tensors="pt",
                       padding=True, truncation=True, max_length=512).to(self.device)
        pred = self.model(**enc).logits.argmax(-1).tolist()
        out = []
        for p in pred:
            l = self.id2label[p]
            out.append("entail" if l.startswith("entail") else "contradict" if l.startswith("contra") else "neutral")
        return out

    def relations(self, parent: str, children: List[str]) -> List[str]:
        """Trả về 'forward-backward', vd. 'entail-neutral' (cha ⇒ con), 'neutral-entail' (con ⇒ cha)."""
        if not children:
            return []
        fw = self._pred([(parent, c) for c in children])
        bw = self._pred([(c, parent) for c in children])
        return [f"{a}-{b}" for a, b in zip(fw, bw)]

nli = NLI(NLI_NAME, DEVICE)
print(nli.relations("Paris is the capital of France.",
                    ["The capital of France is Paris.", "Berlin is the capital of France.", "Bananas are yellow."]))


In [ ]:
#@title 6. Xây dựng Belief Tree (Sec. 3.2)
UNK_WORDS = [w.lower() for w in [
    "I don't", "Not specified", "cannot be determined", "No Answer", "I do not", "N/A", "No information",
    "I cannot", "I can't", "unable to", "don't know", "enough information", "not provided", "None",
    "Cannot answer", "Unavailable", "unclear", "additional context", "further context",
    "additional information", "further information", "Unable to determine", "No check-worthy",
    "premises applicable", "không có thông tin", "không thể xác định", "không rõ", "không biết"]]

_VI_CHARS = set("ăâđêôơưàảãáạằẳẵắặầẩẫấậèẻẽéẹềểễếệìỉĩíịòỏõóọồổỗốộờởỡớợùủũúụừửữứựỳỷỹýỵ")
_DE_FR_CHARS = set("äöüßçéèêëàâîïôûùœ")

def _lang(s: str) -> str:
    """Heuristic nhận dạng ngôn ngữ (vi / cjk / eu / en) để lọc node con lệch ngôn ngữ."""
    low = s.lower()
    if any("\u4e00" <= ch <= "\u9fff" or "\u3040" <= ch <= "\u30ff" or "\uac00" <= ch <= "\ud7af" for ch in low):
        return "cjk"
    if any(ch in _VI_CHARS for ch in low):
        return "vi"
    if any(ch in _DE_FR_CHARS for ch in low):
        return "eu"
    return "en"

def _same_language(parent: str, child: str) -> bool:
    lp, lc = _lang(parent), _lang(child)
    if lp == lc:
        return True
    # câu tiếng Việt ngắn có thể thiếu dấu -> chấp nhận nếu cả hai không chứa ký tự Âu/CJK
    return lp == "vi" and lc == "en" and len(child.split()) <= 4

def _parse_lines(text: str, prefix: str) -> List[str]:
    out = []
    for line in text.splitlines():
        line = line.strip().lstrip("-*• ").strip()
        m = re.match(rf"^{prefix}\s*\d*\s*[:：]\s*(.+)$", line, flags=re.I)
        if m:
            out.append(m.group(1).strip().strip('"'))
    return out

class TreeGenerator:
    def __init__(self, llm: LLM, nli: NLI, max_layers=2, verbose=False,
                 force_both=False, filter_language=True):
        self.llm, self.nli, self.max_layers, self.verbose = llm, nli, max_layers, verbose
        self.force_both, self.filter_language = force_both, filter_language

    def _filter(self, prems: List[str], parent: str) -> List[str]:
        seen, out = set(), []
        for p in prems:
            p = p.strip()
            if not p or ("[" in p and "]" in p):
                continue
            low = p.lower()
            if any(u in low for u in UNK_WORDS) or low == parent.strip().lower() or low in seen:
                continue
            if len(p) > max(300, 3 * len(parent)):
                if self.verbose:
                    print(f"        x (bỏ, quá dài) {p[:80]}")
                continue
            if self.filter_language and not _same_language(parent, p):
                if self.verbose:
                    print(f"        x (bỏ, lệch ngôn ngữ) {p[:80]}")
                continue
            seen.add(low)
            out.append(p)
        return out

    # Strategy 1 ---------------------------------------------------------
    def decompose(self, s: str) -> List[str]:
        ans = self.llm.generate(DECOMPOSE_PROMPT + "\n\nStatement: " + s, temperature=0.0, max_tokens=300)[0]
        claims = _parse_lines(ans, "Claim")
        if any("no check-worthy" in c.lower() for c in claims):
            return []
        return self._filter(claims, s)

    def select_strategy(self, s: str) -> List[str]:
        if self.force_both:
            return ["logical relation", "perturbation"]
        ans = self.llm.generate(STRATEGY_PROMPT + "\n\nTarget statement: " + s + "\nOutput:", temperature=0.0, max_tokens=20)[0].lower()
        if "both" in ans:
            return ["logical relation", "perturbation"]
        if "perturb" in ans:
            return ["perturbation"]
        if "logic" in ans:
            return ["logical relation"]
        return ["logical relation", "perturbation"]

    # Strategy 2 ---------------------------------------------------------
    def logic_relation(self, s: str) -> List[str]:
        prems = []
        for ans in self.llm.generate(SUPPORT_PROMPT + "\n\nTarget statement: " + s, temperature=0.7, max_tokens=400, n=2):
            prems += _parse_lines(ans, "Premise")
        prems = self._filter(prems, s)
        if not prems:
            for ans in self.llm.generate(CONTRADICT_PROMPT + "\n\nTarget statement: " + s, temperature=0.7, max_tokens=400, n=2):
                prems += _parse_lines(ans, "Premise")
            prems = self._filter(prems, s)
        if not prems:
            return []
        rels = self.nli.relations(s, prems)
        prems = [p for p, r in zip(prems, rels) if r != "neutral-neutral"]
        return prems[:3]

    # Strategy 3 ---------------------------------------------------------
    def correction(self, s: str) -> List[str]:
        ans = self.llm.generate(QUESTION_PROMPT + "\n\nStatement: " + s, temperature=0.0, max_tokens=80)[0]
        qs = _parse_lines(ans, "Question")
        if not qs:
            return []
        answers = self.llm.generate(qs[0], temperature=0.7, max_tokens=120, n=5)
        revised = []
        for a in answers:
            out = self.llm.generate(REVISE_PROMPT.format(background=a, statement=s), temperature=0.0, max_tokens=120)[0]
            r = _parse_lines(out, r"Revised\s*(?:Answer|Statement)")
            if r:
                revised.append(r[0])
        return self._filter(revised, s)[:5]

    # Expand one node -----------------------------------------------------
    def expand(self, s: str, can_decompose: bool) -> List[Dict]:
        if can_decompose:
            subs = self.decompose(s)
            if len(subs) > 1:
                return [{"method": "decompose", "children": subs, "rels": ["decompose"] * len(subs)}]
        groups = []
        for strat in self.select_strategy(s):
            kids = self.logic_relation(s) if strat == "logical relation" else self.correction(s)
            if not kids:
                continue
            rels = self.nli.relations(s, kids)
            keep = [(k, r) for k, r in zip(kids, rels) if r != "neutral-neutral"]
            if keep:
                groups.append({"method": strat, "children": [k for k, _ in keep], "rels": [r for _, r in keep]})
        return groups

    # BFS build ------------------------------------------------------------
    def build(self, root: str, root_can_decompose=True) -> Dict[str, List[Dict]]:
        tree: Dict[str, List[Dict]] = {}
        queue = [(root, 1, root_can_decompose)]
        visited = set()
        while queue:
            s, layer, can_dec = queue.pop(0)
            if s in visited or layer > self.max_layers:
                continue
            visited.add(s)
            groups = self.expand(s, can_dec)
            if self.verbose:
                for g in groups:
                    print(f"  [L{layer}] {g['method']:<16} <- {s[:70]}")
                    for c, r in zip(g["children"], g["rels"]):
                        print(f"        - ({r}) {c[:90]}")
            tree[s] = groups
            for g in groups:
                if g["method"] == "perturbation":
                    continue          # node sửa lỗi không mở rộng tiếp (homogeneous với cha)
                for c in g["children"]:
                    queue.append((c, layer + 1, False))
        return tree


In [ ]:
#@title 7. Hidden Markov Tree & Belief Propagation (Sec. 3.3 – 3.5)
HMM_CONFIG = {
    "prior_true": 0.5, "prior_false": 0.5,
    "bins_sep": [0.19, 0.39, 0.69, 0.89],
    # Table 1 của paper – ước lượng trên Wikibio-GPT3
    "emission_true":  [0.1190, 0.0516, 0.0992, 0.0833, 0.6468],
    "emission_false": [0.2971, 0.0957, 0.1557, 0.1343, 0.3171],
    # node từ statement correction có S_u = 1 cố định (Appendix A.1)
    "perturb_emission_true": 0.8, "perturb_emission_false": 0.2,
    "p_t": 0.5, "p_f": 0.5,
}

class HMMTreeNode:
    def __init__(self, statement: str, conf: float, gen_method: str = "root", layer=0, cfg=HMM_CONFIG):
        self.statement, self.conf, self.gen_method, self.layer, self.cfg = statement, conf, gen_method, layer, cfg
        self.children: List["HMMTreeNode"] = []
        self.nli_rels: List[str] = []
        self.group_methods: List[str] = []
        self.posterior = None

    def add_child(self, child: "HMMTreeNode", nli_rel: str, method: str):
        self.children.append(child); self.nli_rels.append(nli_rel); self.group_methods.append(method)

    # p(S_u | Z_u) ----------------------------------------------------------
    def emission(self, z: bool) -> float:
        c = self.cfg
        if self.gen_method == "perturbation":
            return c["perturb_emission_true"] if z else c["perturb_emission_false"]
        idx = sum(self.conf >= b for b in c["bins_sep"])
        return (c["emission_true"] if z else c["emission_false"])[idx]

    # p(Z_v | Z_u) theo Table 2 ---------------------------------------------
    def transition(self, rel: str, zu: bool, zv: bool) -> float:
        c = self.cfg
        if zu:
            if rel in ("entail-entail", "entail-neutral"):       # u ⇔ v hoặc u ⇒ v
                return 1.0 if zv else 0.0
            if "contradict" in rel:                               # u ⇒ ¬v
                return 0.0 if zv else 1.0
            return c["p_t"] if zv else c["p_f"]                   # u ⇐ v
        else:
            if rel in ("entail-entail", "neutral-entail"):        # u ⇔ v hoặc u ⇐ v
                return 0.0 if zv else 1.0
            return c["p_t"] if zv else c["p_f"]                   # u ⇒ v, u ⇒ ¬v

    # β(z,u) = p(S_T(u) | Z_u = z)  (Eq. 1) -----------------------------------
    def beta(self, z: bool) -> float:
        total = self.emission(z)
        if not self.children:
            return total
        if self.group_methods[0] == "decompose":
            bt = [ch.beta(True) for ch in self.children]
            bf = [ch.beta(False) for ch in self.children]
            if z:
                prod = 1.0
                for b in bt: prod *= b
                total *= prod
            else:
                m = len(self.children)
                combos = list(itertools.product([False, True], repeat=m))[:-1]   # loại tổ hợp toàn True
                acc = 0.0
                for combo in combos:
                    p = 1.0 / len(combos)
                    for i, s in enumerate(combo):
                        p *= bt[i] if s else bf[i]
                    acc += p
                total *= acc
        else:
            for ch, rel in zip(self.children, self.nli_rels):
                total *= (ch.beta(True) * self.transition(rel, z, True) +
                          ch.beta(False) * self.transition(rel, z, False))
        return total * 10  # tránh underflow, không ảnh hưởng posterior

    def get_posterior(self) -> float:
        jt = self.beta(True) * self.cfg["prior_true"]
        jf = self.beta(False) * self.cfg["prior_false"]
        self.posterior = jt / (jt + jf + 1e-15)
        return self.posterior

    def pretty(self, indent=0) -> str:
        pad = "  " * indent
        s = f"{pad}[{self.gen_method}] conf={self.conf:.2f} :: {self.statement}\n"
        for ch, rel in zip(self.children, self.nli_rels):
            s += f"{pad}  └─({rel})\n" + ch.pretty(indent + 2)
        return s


In [ ]:
#@title 8. BTProp end-to-end cho một mệnh đề
class BTProp:
    def __init__(self, llm: LLM, nli: NLI, max_layers=2, verbose=False):
        self.llm, self.nli, self.verbose = llm, nli, verbose
        self.gen = TreeGenerator(llm, nli, max_layers, verbose,
                                 force_both=FORCE_BOTH_STRATEGIES, filter_language=FILTER_LANGUAGE)
        self.result_cache: Dict[str, Dict] = {}

    def score(self, statement: str, can_decompose=True) -> Dict:
        statement = statement.strip()
        if statement in self.result_cache:
            return self.result_cache[statement]
        if self.verbose:
            print(f"\n=== Belief tree cho: {statement}")
        tree = self.gen.build(statement, can_decompose)
        prior = self.llm.confidence(statement)
        root = HMMTreeNode(statement, prior)
        index = {statement: root}
        for parent_s, groups in tree.items():
            parent = index.get(parent_s)
            if parent is None:
                continue
            for g in groups:
                for c, r in zip(g["children"], g["rels"]):
                    if c in index:
                        continue
                    conf = 1.0 if g["method"] == "perturbation" else self.llm.confidence(c)
                    node = HMMTreeNode(c, conf, g["method"], parent.layer + 1)
                    parent.add_child(node, r, g["method"])
                    index[c] = node
        post = root.get_posterior() if root.children else prior
        if self.verbose:
            print(root.pretty())
            print(f"  prior={prior:.3f}  ->  posterior={post:.3f}")
        res = {"statement": statement, "prior": prior, "posterior": post, "n_nodes": len(index), "tree": root}
        self.result_cache[statement] = res
        return res

btprop = BTProp(llm, nli, MAX_LAYERS, VERBOSE)


In [ ]:
#@title 9. Nâng cấp: JSON câu hỏi trắc nghiệm → mức độ ảo giác
W_CORRECT, W_DISTRACTOR, W_EXPLAIN = 0.4, 0.3, 0.3   # trọng số tổng hợp
MAX_EXPLAIN_CLAIMS = 4

def is_true_flag(v) -> bool:
    return str(v).strip().lower() in {"true", "1", "yes", "y", "t", "đúng", "correct", "x"}

def option_to_statement(llm: LLM, question: str, option: str) -> str:
    out = llm.generate(MCQ_TO_STATEMENT_PROMPT + f"\n\nQuestion: {question}\nOption: {option}", temperature=0.0, max_tokens=80)[0]
    st = _parse_lines(out, "Statement")
    return st[0] if st else f"{question.rstrip('?？ ')} : {option}"

def to_english(llm: LLM, text: str) -> str:
    if not PIVOT_TO_ENGLISH or _lang(text) == "en":
        return text
    out = llm.generate(TRANSLATE_PROMPT + f"\n\nSentence: {text}", temperature=0.0, max_tokens=120)[0]
    en = _parse_lines(out, "English")
    return en[0] if en else text

def level(h: float) -> str:
    return "THẤP" if h < 0.3 else "TRUNG BÌNH" if h < 0.6 else "CAO"

def evaluate_mcq(data: Dict, btprop: BTProp) -> Dict:
    q = data["content"].strip()
    options = data.get("options", [])
    explanation = (data.get("explanation") or "").strip()

    report = {"correct": [], "distractors": [], "explanation": []}
    for opt in options:
        st = option_to_statement(btprop.llm, q, opt["text"])
        st_en = to_english(btprop.llm, st)
        res = btprop.score(st_en, can_decompose=False)
        item = {"key": opt.get("key"), "text": opt["text"], "statement": st, "statement_en": st_en,
                "prior": res["prior"], "posterior": res["posterior"]}
        (report["correct"] if is_true_flag(opt.get("is_correct")) else report["distractors"]).append(item)

    if explanation:
        expl_work = to_english(btprop.llm, explanation)
        claims = btprop.gen.decompose(expl_work)[:MAX_EXPLAIN_CLAIMS] or [expl_work]
        for c in claims:
            c_en = to_english(btprop.llm, c)   # không đổi nếu đã là tiếng Anh
            res = btprop.score(c_en, can_decompose=False)
            report["explanation"].append({"claim": c, "claim_en": c_en, "prior": res["prior"], "posterior": res["posterior"]})

    parts, weights = [], []
    if report["correct"]:
        h_c = 1 - min(x["posterior"] for x in report["correct"])
        parts.append(h_c); weights.append(W_CORRECT)
    else:
        h_c = None
    if report["distractors"]:
        h_d = max(x["posterior"] for x in report["distractors"])
        parts.append(h_d); weights.append(W_DISTRACTOR)
    else:
        h_d = None
    if report["explanation"]:
        h_e = 1 - sum(x["posterior"] for x in report["explanation"]) / len(report["explanation"])
        parts.append(h_e); weights.append(W_EXPLAIN)
    else:
        h_e = None
    score = sum(p * w for p, w in zip(parts, weights)) / sum(weights) if weights else 0.5
    report.update({"h_correct": h_c, "h_distractor": h_d, "h_explanation": h_e,
                   "hallucination_score": score, "level": level(score)})
    return report

def print_report(r: Dict):
    print("\n" + "=" * 70)
    print("KẾT QUẢ ĐÁNH GIÁ MỨC ĐỘ ẢO GIÁC (BTProp)")
    print("=" * 70)
    def _en(x, k):
        return f"  (EN: {x[k]})" if x.get(k) and x[k] != x[k.replace("_en", "")] else ""
    for x in r["correct"]:
        print(f"[Đáp án đúng {x['key']}] {x['statement']}{_en(x, 'statement_en')}")
        print(f"    prior={x['prior']:.3f}  posterior P(đúng)={x['posterior']:.3f}  ->  ảo giác={1-x['posterior']:.3f}")
    for x in r["distractors"]:
        print(f"[Đáp án sai  {x['key']}] {x['statement']}{_en(x, 'statement_en')}")
        print(f"    prior={x['prior']:.3f}  posterior P(đúng)={x['posterior']:.3f}  ->  ảo giác={x['posterior']:.3f}")
    for x in r["explanation"]:
        print(f"[Giải thích] {x['claim']}{_en(x, 'claim_en')}")
        print(f"    prior={x['prior']:.3f}  posterior P(đúng)={x['posterior']:.3f}  ->  ảo giác={1-x['posterior']:.3f}")
    print("-" * 70)
    fmt = lambda v: "n/a" if v is None else f"{v:.3f}"
    print(f"h_correct={fmt(r['h_correct'])}  h_distractor={fmt(r['h_distractor'])}  h_explanation={fmt(r['h_explanation'])}")
    print(f"\n>>> MỨC ĐỘ ẢO GIÁC: {r['hallucination_score']:.3f}  ({r['level']})")
    print("=" * 70)


In [ ]:
#@title 10. Nhập JSON và chạy
import json
EXAMPLE_JSON = r"""{
  "content": "Thủ đô của Pháp là thành phố nào?",
  "options": [
    {"key": "A", "text": "Berlin", "is_correct": "false"},
    {"key": "B", "text": "Paris",  "is_correct": "true"},
    {"key": "C", "text": "Madrid", "is_correct": "false"},
    {"key": "D", "text": "Rome",   "is_correct": "false"}
  ],
  "explanation": "Paris là thủ đô và thành phố lớn nhất của Pháp, nằm bên bờ sông Seine."
}"""

raw = input("Dán chuỗi JSON câu hỏi (Enter để dùng ví dụ mẫu): ").strip()
data = json.loads(raw or EXAMPLE_JSON)

report = evaluate_mcq(data, btprop)
print_report(report)


## Ghi chú cài đặt

- **Confidence (prior belief)**: xác suất token kế tiếp `True` vs `False` sau prompt *"True or False?"* (Sec. 3.2), chuẩn hoá softmax giữa hai token – tương đương `confidence_estimation.py` của repo gốc.
- **Belief tree**: 3 chiến lược sinh node con như paper (`decompose` → `logical relation` / `perturbation`), độ sâu tối đa `MAX_LAYERS`; node từ *statement correction* có confidence cố định 1.0 và emission riêng (0.8/0.2).
- **Quan hệ logic**: NLI hai chiều → `entail-entail` (⇔), `entail-neutral` (⇒), `neutral-entail` (⇐), có `contradict` (⇒¬); `neutral-neutral` bị loại.
- **Inference**: thuật toán upward (Eq. 1) với transition theo Table 2 và emission theo Table 1; nhánh `decompose` dùng phân phối đồng thời (ít nhất một con sai khi cha sai).
- **Nâng cấp cho MCQ**: ảo giác = tổ hợp của (i) đáp án đúng bị model tin là sai, (ii) đáp án sai bị model tin là đúng, (iii) các claim trong lời giải thích không đáng tin.
- `FORCE_BOTH_STRATEGIES = True`: luôn sinh node con bằng cả *logical relation* lẫn *statement correction* (cây giàu hơn, chậm hơn ~2×). Tắt để LLM tự chọn như paper.
- `PIVOT_TO_ENGLISH = True`: dịch mệnh đề sang tiếng Anh trước khi chạy BTProp (giữ nguyên văn khi in). Lý do: backbone 7B kém ổn định về kiến thức khi suy luận bằng tiếng Việt (vd. chấm *"Sao Mộc là hành tinh lớn nhất"* là sai), trong khi paper và emission Table 1 đều được hiệu chỉnh trên tiếng Anh.
- `FILTER_LANGUAGE = True`: loại node con sinh ra bằng ngôn ngữ khác câu gốc (chống rò ngôn ngữ của LLM).
- Để chạy nhanh hơn: đặt `MAX_LAYERS = 1`, tắt `FORCE_BOTH_STRATEGIES` hoặc dùng `Qwen/Qwen2.5-3B-Instruct`.
